# Vortex Field Solver Resilience
This notebook provides a detailed, mathematically rigorous walkthrough of the **Holonomic Association Model (HAM)**.
The HAM library treats differentiable **Finsler Geometries** as learnable modules in JAX, allowing us to solve optimal control and simulation problems purely through geometric dynamics.

### Core Philosophy: Metric-First Design
In HAM, the `FinslerMetric` object is the single source of truth. We define the scalar **Energy Functional**:
$$\mathcal{E}[\gamma] = \int \frac{1}{2} F^2(x, \dot{x}) dt$$
where $F(x,v)$ is the Finsler cost function. Everything else—including the Geodesic Spray (the physics engine), curvature, and Berwald parallel transport—is auto-differentiated directly from $F$ using `jax.grad` and `jax.hessian`, entirely avoiding the $O(N^3)$ computational cost of expanding Christoffel symbols manually.

## 1. Setup and Imports
The `demo_vortex` notebook illustrates the resilience of the **Augmented Vertex Block Descent (AVBD)** solver when faced with extreme, non-linear geometric sprays. We use Plotly for interactive 3D rendering.


In [1]:
from ham.utils.config import DEFAULT_JNP_DTYPE, DEFAULT_NP_DTYPE
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
import numpy as np
from jax import config

from ham.geometry import Sphere, Randers
from ham.solvers import AVBDSolver
from ham.vis import generate_icosphere


## 2. Modeling the Vortex Field
We define a highly localized rotational vector field with exponential decay. The extreme gradients in this wind field test the numerical stability of the implicit dynamics underlying the AVBD optimization.


In [2]:
def vortex_field(center, strength=1.0, decay=2.0):
    center = center / jnp.linalg.norm(center)
    def flow(x):
        cos_dist = jnp.dot(x, center)
        dist = jnp.arccos(jnp.clip(cos_dist, -1.0, 1.0))
        v_rot = jnp.cross(center, x)
        magnitude = strength * jnp.exp(-decay * (dist**2))
        return magnitude * v_rot
    return flow


## 3. Environment Definition
We construct the `Randers` metric using Zermelo's Navigation formula. Notice the incredibly high `strength=10.0` which forces extreme path curvature in order to minimize travel time through the heavy headwind.


In [3]:
sphere = Sphere(radius=1.0)
vortex_center = jnp.array([0.0, 1.0, 0.0])
w_net = vortex_field(vortex_center, strength=10.0, decay=5.0)
h_net = lambda x: jnp.eye(3)
metric = Randers(sphere, h_net, w_net)

start = jnp.array([1.0, 0.0, 0.0])
end   = jnp.array([-0.99, 0.1, 0.0]) 
end   = end / jnp.linalg.norm(end)


## 4. Solver Stiffness Comparison (`beta` parameter)
The AVBD solver incorporates a `beta` penalty that controls block descent stiffness. 
- **High Beta (Stiff)**: Quickly freezes the path, behaving greedily and potentially getting trapped in local topological minima.
- **Low Beta (Relaxed)**: Allows massive lateral relaxation along the manifold to explore topologically distinct routes around the vortex core.


In [4]:
print("Solving 'Stiff' (High Beta)...")
solver_stiff = AVBDSolver(step_size=0.05, beta=20.0, iterations=100)
traj_stiff = solver_stiff.solve(metric, start, end, n_steps=40)

print("Solving 'Relaxed' (Low Beta)...")
solver_relaxed = AVBDSolver(step_size=0.05, beta=0.5, iterations=100) 
traj_relaxed = solver_relaxed.solve(metric, start, end, n_steps=40)


Solving 'Stiff' (High Beta)...
Solving 'Relaxed' (Low Beta)...


## 5. Visualizing the Vortex Escape in 3D
The interactive visualization clearly shows the relaxed AVBD solver dynamically shifting the vertices globally to discover a much more efficient topological route (the 'D' shape) avoiding the worst of the headwind. Rotate the render to see how the red path swoops around the vortex core!


In [6]:
fig = go.Figure()

pts, faces = generate_icosphere(radius=1.0, subdivisions=4)
wind_vecs = np.array(jax.vmap(w_net)(pts))

# Plot Manifold
fig.add_trace(go.Mesh3d(
    x=pts[:,0], y=pts[:,1], z=pts[:,2],
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    color='lightblue', opacity=0.2, alphahull=0, name='Sphere'
))

# Plot Vortex Vectors
# Subsample to keep the plot clean
mask = np.linalg.norm(wind_vecs, axis=1) > 0.5
fig.add_trace(go.Cone(
    x=pts[mask,0], y=pts[mask,1], z=pts[mask,2],
    u=wind_vecs[mask,0], v=wind_vecs[mask,1], w=wind_vecs[mask,2],
    sizemode='absolute', sizeref=0.3, colorscale='Viridis', showscale=True, name='Vortex'
))

# Plot Paths
fig.add_trace(go.Scatter3d(
    x=traj_stiff.xs[:,0], y=traj_stiff.xs[:,1], z=traj_stiff.xs[:,2],
    mode='lines+markers', line=dict(color='gray', width=4, dash='dash'), 
    marker=dict(size=3), name='Stiff Solver (beta=20)'
))
fig.add_trace(go.Scatter3d(
    x=traj_relaxed.xs[:,0], y=traj_relaxed.xs[:,1], z=traj_relaxed.xs[:,2],
    mode='lines+markers', line=dict(color='red', width=6), 
    marker=dict(size=3), name='Relaxed Solver (beta=0.5)'
))

fig.update_layout(title="AVBD Solver Stiffness Comparison (Rotate to explore!)", scene=dict(aspectmode='data'))
fig.show()
